## **Installing and importing necessary libraries**

In [ ]:
!pip install -q emoji emoticon_fix transformers

In [ ]:
# torch- needed for tensors, datasets, and model training

import numpy as np
import pandas as pd
import re, emoji, json, torch
from torch import nn
from torch.optim import AdamW
from emoticon_fix import emoticon_fix
from torch.utils.data import DataLoader
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification

## **Load the updated (null-valued rows dropped) dataset**

In [ ]:
df = pd.read_csv("Mental_Distress_Dataset_updated.csv")

## **Text preprocessing**

---

*   Converts text to lowercase first
*   Removes extra space from text
*   Convert emojis and emoticons to textual form (😉 > :wink:)
*   Removes mentions from text starting with @ (@user)


In [ ]:
def preprocess_text(text):
  text = text.lower()
  """
  '\s+'- matches miltiple spaces/tabs/newlines, replacing with one space only
  strip() helps to remove spaces at the beginning and end
  """
  text = re.sub(r'\s+', ' ', text).strip()
  text = emoji.demojize(text)
  text = emoticon_fix(text)
  text = re.sub(r'@\w+', '', text) #  # @\w+ matches things like @john

  return text

# Apply preprocessing
df['text'] = df['text'].apply(preprocess_text)

## **Assigning numbers to the labels and saving them as JSON for reuse (run this code if it's the first time)**

In [ ]:
le = LabelEncoder()
df['label_encoded'] = le.fit_transform(df['label'])

In [ ]:
df['label_encoded']

In [ ]:
encoded_labels_to_check = [0, 1, 2, 3, 4]

original_labels = le.inverse_transform(encoded_labels_to_check)

print("Original labels for encoded values 0, 1, 2, 3, 4:")
for encoded, original in zip(encoded_labels_to_check, original_labels):
    print(f"Encoded: {encoded} -> Original: {original}")

In [ ]:
label_mapping = dict(zip(le.classes_, le.transform(le.classes_)))
print(label_mapping) # numpy.int64 values which JSON can't save properly

In [ ]:
# Convert numpy.int64 values to standard Python int for JSON serialization
serializable_label_mapping = {key: int(value) for key, value in label_mapping.items()}

with open("label_mapping.json", "w") as f:
    json.dump(serializable_label_mapping, f, indent=4)

In [ ]:
# with open("label_mapping.json", "r") as f:
#     label_mapping = json.load(f)

# id_to_label = {v: k for k, v in label_mapping.items()}

## **Assign a new column for that labels**

In [ ]:
# df['label_encoded'] = df['label'].map(label_mapping)

## **Train-test stratified split**

In [ ]:
"""
stratify=df['label_encoded'] ensures that the train and test
sets have the same label distribution as the original dataset
"""

train_texts, test_texts, train_labels, test_labels = train_test_split(
    df['text'],
    df['label_encoded'],
    test_size=0.2,
    stratify=df['label_encoded'],
    random_state=42
)

In [ ]:
df_test = pd.DataFrame({
    'text': test_texts,
    'label_encoded': test_labels
})

In [ ]:
df_test.to_csv("mental_distress_test_set.csv", index=False)

# **Transformer Model (DistilBERT)**

## **Load DistilBERT tokenizer**

In [ ]:
# Converts text → tokens → numbers
tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")

## **Transforming text to numbers for training**

In [ ]:
"""
converts raw text → numerical format that DistilBERT can understand
"""

train_encodings = tokenizer(
    list(train_texts), # Converts pandas column into a Python list of strings
    truncation=True, # Max length = 128. If text has 200 tokens → keep first 128 (prevents memory issue)
    padding=True, # Makes all sequences (ordered list of tokens (numbers)) the same length
    max_length=256 # maximum number of tokens per input
)

test_encodings = tokenizer(
    list(test_texts),
    truncation=True,
    padding=True,
    max_length=256
)

## **DistilBERT custom dataset**

This `MentalDataset` custom class **wraps tokenized text and labels into PyTorch-friendly tensors**, allowing the model to efficiently access, batch, and train on the data. It ensures each sample includes both inputs (`input_ids`, `attention_mask`) and the corresponding label for loss computation.

In [ ]:
# Custom dataset class- required for PyTorch training

"""
class MentalDataset(torch.utils.data.Dataset):
  custom dataset class by inheriting from torch.utils.data.Dataset
  PyTorch expects datasets to be in this format.
  It allows the model to read data efficiently in batches.

__init__ is the constructor
"""
class MentalDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings # tokenized text from DistilBERT tokenizer
        self.labels = labels # numeric labels for each post (label_encoded)

    # tells PyTorch how to access a single data sample
    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels.iloc[idx])
        return item

    # Returns the total number of samples in the dataset
    def __len__(self):
        return len(self.labels)

## **Create separate objects for both train and test set**

In [ ]:
train_dataset = MentalDataset(train_encodings, train_labels)
test_dataset = MentalDataset(test_encodings, test_labels)

## **Class weights**

In [ ]:
classes = np.unique(train_labels) # finds all unique labels in your training set
class_weights = compute_class_weight( # automatically rare classes get higher weight, common classes get lower weight
    class_weight='balanced',
    classes=classes,
    y=train_labels
)

"""
converts the weights into a PyTorch tensor so it can be used in
the loss function (CrossEntropyLoss) during training.
"""
class_weights = torch.tensor(class_weights, dtype=torch.float)
print(class_weights)

In [ ]:
# class_weights_np = class_weights.numpy()
# print(class_weights_np)

## **Load DistilBERT model**

In [ ]:
"""
DistilBertForSequenceClassification- predefined class from Hugging Face Transformers
DistilBERT + a classification head (MLP)
"""
model = DistilBertForSequenceClassification.from_pretrained(
    'distilbert-base-uncased',
    num_labels=len(classes)
)

## **Dataloader**

*   wraps your dataset and makes it easy to feed data into the model in small batches
*   how data is fed
*   optimizer helps to make the model learn by updating its weights
*   as the original dataset is large, all the data can't be send at once. So, we split into batches
*   splits into batches of `batch size` samples. For example, if 16, it means that model sees 16 posts at a time.
*   Small batch → slower but stable
*   Large batch → faster but memory heavy [trade-offs]


In [ ]:
# 16 batch size fits in memory properly and training becomes stable

"""
shuffle=True:
  Randomizes data every epoch
  Prevents model from learning patterns like 'first 100 samples = same class'
  Improves generalization and reduces overfitting
  No need for test_loader as we evaluate on unseen data using this set
"""

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32)

## **Optimizer**

In [ ]:
# Optimizer helps updating model weights after each batch
# When a model makes a mistake, optimizer fixes it

# model.parameters()- all weights and biases inside DistilBERT that get updated during training

"""
learning rate:
  too high- unstable learning
  too low- very slow learning
"""
# AdamW- more stable than basic optimizers

optimizer = AdamW(model.parameters(), lr=3e-5)

## **Loss Function**

In [ ]:
"""
Loss- how wrong the model is
  high loss- bad prediction
  low loss- good prediction!!

CrossEntropyLoss- it measures how well the model predicts the correct class in a multi-class classification problem

weight=class_weights- class balancing strategy
  Class weight will help penalizing rare classes more than common classes
  Instead of changing data (oversampling), we change importance of errors in this way
"""
loss_fn = torch.nn.CrossEntropyLoss(weight=class_weights)

In [ ]:
# torch.cuda.is_available()- is GPU available?

device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
model.to(device)

In [ ]:
# Batch → Prediction → Loss → Backprop → Update → Repeat

epochs = 4 # the number of times the model sees the entire dataset

for epoch in range(epochs):
    model.train() # model in training mode
    total_loss = 0

    for batch in train_loader:
        optimizer.zero_grad()

        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        # Forward pass
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss
        total_loss += loss.item() # Tracks loss

        loss.backward() # Backpropagation (computes gradients)
        optimizer.step() # uses gradients to adjust weights and improve model

    # Shows average loss per epoch
    print(f"Epoch {epoch+1}/{epochs} - Loss: {total_loss/len(train_loader):.4f}")

In [ ]:
model.eval() # switches model to evaluation mode
preds, true_labels = [], [] # store model predictions and actual labels

# no_grad- OFF gradient calculation
with torch.no_grad():
    for batch in test_loader: # goes batch by batch
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        # Forward pass. no labels just to give predictions, not compute loss
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits # Raw scores before probability

        predictions = torch.argmax(logits, dim=-1)

        preds.extend(predictions.cpu().numpy())
        true_labels.extend(labels.cpu().numpy())

# Ensure labels are in correct order
target_names = [id_to_label[i] for i in sorted(id_to_label.keys())]

print(classification_report(true_labels, preds, target_names=target_names, digits=4))

## **Stress Score Mapping**

In [ ]:
"""
Numeric labels to textual labels to stress scores
"""

# Converts emotion labels → numeric stress values
stress_mapping = {
    'Suicidal': 5,
    'Depressed': 4,
    'Anxious': 3,
    'Frustrated': 2,
    'Others': 1
}

# Convert model predictions (numeric) → stress scores
pred_labels = [id_to_label[i] for i in preds]  # preds from previous step
stress_scores = [stress_mapping[label] for label in pred_labels] # pred_labels - DistilBERT predictions

## **Batch Average**

In [ ]:
batch_size = 16  # same as training
aggregated_scores = []

# main computation
def aggregate_stress_scores(scores, batch_size=16):
    return [np.mean(scores[i:i+batch_size]) for i in range(0, len(scores), batch_size)]

aggregated_scores = aggregate_stress_scores(stress_scores)
print(aggregated_scores[:5])

## **Normalizing scores**

In [ ]:
# Normalize to 0-10 scale (main function)

def normalize_score(score, min_score=1, max_score=5):
    return (score - min_score) / (max_score - min_score) * 10

## **Stress level mapping**

In [ ]:
# 0–4 → Low, 4–7 → Medium, 7–10 → High

def get_stress_level(score):
    if score <= 4:
        return "Low"
    elif score <= 7:
        return "Medium"
    else:
        return "High"

## **Input prediction function**

In [ ]:
# Function to predict stress for a single user input
def predict_user_input(text):

    # Preprocess the input text
    processed_text = preprocess_text(text)

    # Tokenize
    encoding = tokenizer(
        [processed_text],
        truncation=True,
        padding=True,
        max_length=256,
        return_tensors='pt'  # returns PyTorch tensors
    )

    # Move to device
    input_ids = encoding['input_ids'].to(device)
    attention_mask = encoding['attention_mask'].to(device)

    # Model prediction
    model.eval()
    with torch.no_grad():
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits
        pred_id = torch.argmax(logits, dim=-1).item()  # single prediction

    # Map prediction to label
    pred_label = id_to_label[pred_id]

    stress_score = stress_mapping[pred_label]   # label → number
    normalized_score = normalize_score(stress_score)  # number → scaled
    stress_level = get_stress_level(normalized_score) # scaled → category

    return pred_label, stress_score, normalized_score, stress_level

In [ ]:
# List to store stress scores for all user inputs
user_stress_scores = []

num_posts = int(input("How many posts do you want to input? "))

"""
4 tasks inside the function:
  Take input
  Predict
  Print result
  Store scores
"""
for i in range(num_posts):
    user_text = input(f"Enter post {i+1}: ")

    label, score, norm_score, level = predict_user_input(user_text)

    print(f"Post {i+1}: {label} | Score: {score} | Normalized: {norm_score:.2f} | Level: {level}\n")

    user_stress_scores.append(score)

# Aggregate user inputs
batch_avg = np.mean(user_stress_scores)

# Normalize aggregated scores
normalized_batch_score = normalize_score(batch_avg)

# Final stress level
batch_stress_level = get_stress_level(normalized_batch_score)

print(f"\nAggregated Stress Score (normalized 0-10): {normalized_batch_score:.2f}")
print(f"Aggregated Stress Level for your posts: {batch_stress_level}")

In [ ]:
torch.save(model.state_dict(), "distilbert_emotion_model.pth")